# **Collaborative Filtering Recommendation System (V3) KNN + Popularity Based**

## KNN
- This is an item-item collaborative filtering approach
- Instead of clustering users, we cluster books based on their rating patterns
- For each book, we create a similarity metric that measures how similar it is to other books
- Which other books have rating patterns across users most similar to this liked book?
- Books with similar rating patterns are considered neighbors and are candidates for recommendation

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## **Data Laoder**

In [4]:
books = pd.read_csv("data/BX-Books.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)
users = pd.read_csv("data/BX-Users.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)
ratings = pd.read_csv("data/BX-Book-Ratings.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)

## **Data Preprocessing**

### Book data processing
- column name refactoring
- choose needed columns only
- quick preview

In [5]:
books.shape

(271360, 8)

In [6]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [7]:
books.head(3)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...


In [8]:
# Choose needed columns only
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-L']]

# renaming those into model friendly way
books.rename(columns = {
    "ISBN": "isbn",
    "Book-Title": "title",
    "Book-Author": "author",
    "Year-Of-Publication": "year",
    "Publisher": "publisher",
    "Image-URL-L": "img_url"
}, inplace=True)

books.head(3)

,isbn,title,author,year,publisher,img_url
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...


### Users data processing
- column name refactoring
- choosed all columns
- quick preview

In [9]:
users.shape

(278858, 3)

In [10]:
users.head(3)

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN


In [11]:
# Here we need all columns, let rename them

users.rename(columns = {
    "User-ID": "user_id",
    "Location": "location",
    "Age": "age"
}, inplace=True)

users.head(3)

,user_id,location,age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN


### Ratings data processing
- column name refactoring
- choose all columns
- quick preview

In [12]:
ratings.shape

(1149780, 3)

In [13]:
ratings.columns

Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='object')

In [14]:
ratings.head(3)

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0


In [15]:
# Here we need all columns, let rename them

ratings.rename(columns = {
    "User-ID": "user_id",
    "ISBN": "isbn",
    "Book-Rating": "ratings"
}, inplace=True)

ratings.head(3)

,user_id,isbn,ratings
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0


## **Data Filtering**

### Function to Build Dataset with Parameters

In [16]:
# min_user_ratings=200 > keep only users who rated more than 200 books
# min_book_ratings=50 > keep only books that have more than 50 ratings count

from sklearn.model_selection import train_test_split

def build_dataset(min_user_ratings=200, min_book_ratings=50):

    # filter active users ( we keep only users who rated more than min_user_ratings times )
    user_counts = ratings['user_id'].value_counts() # Count how many times each user rated (index = user_id, value = rating count )
    active_users_mask = user_counts > min_user_ratings # Create True/False mask
    active_users = user_counts[active_users_mask].index # Keep only users with more than min_user_ratings ratings (gives only the user_ids)
    filtering_mask = ratings['user_id'].isin(active_users) # Is each value inside ratings['user_id'] present in active_users? returns True/False Boolean Series
    filtered_ratings  = ratings[filtering_mask] # kkep only values that was true in filtering_mask series
    rating_with_books = filtered_ratings.merge(books, on = "isbn") # Combine books & filtered_reatings on isbn, only keep records that both have same isbn
    
    # filter popular books ( we keep only books that rated by users more than min_book_ratings times )
    book_counts = rating_with_books['title'].value_counts() # Count how many times each book appear in table (how many times rated by a user)
    popular_books_mask = book_counts > min_book_ratings # mask that return boolean series by condition each value > min_book_ratings
    popular_books = book_counts[popular_books_mask].index # apply above mask, return index (=title) of book that True in popular_books_mask
    title_intersection_mask = rating_with_books["title"].isin(popular_books) # Boolean series, T = titles apear in both filtered_ratings & popular_books
    rating_with_books = rating_with_books[title_intersection_mask] # Books only was in popular_books

    # Dropping duplicates
    rating_with_books.drop_duplicates(['user_id', 'title'], inplace=True)
    # print("len : ", len(rating_with_books.groupby("user_id"))) # number of groups / Number of unique users in final_ratings
    
    # train/test split
    train_list, test_list = [], []

    # groupby("user_id") splits the rating_with_books dataframe into mini dataframes per user
    # group_dataframe = ratings of ONE user, all ratings of that user (DataFrame)
    # user : user_id value
    for user, group_dataframe in rating_with_books.groupby("user_id"):
        
        # Users with very few ratings are skipped
        if len(group_dataframe) < 5:
            continue
        
        # we have only passed one data set that will devide usually into 2, if we passes two as (X, y, ...) then it x --> 2, y --> that is why we see 4 
        # variables as X_train, X_test, y_train, y_test
        # 80% data frames for tratining
        train, test = train_test_split(group_dataframe, test_size=0.2, random_state=42) 
        train_list.append(train)
        test_list.append(test)
    
    # pd.concat() = stack/merge each users group dataframes vertically (row-wise).
    train_ratings = pd.concat(train_list)
    test_ratings  = pd.concat(test_list)
    
    # print("train_ratings : ", train_ratings.shape)
    # print("test_ratings : ", test_ratings.shape)

    # Build Pivot ONLY from Train Data
    # KNN finds similarity between ROWS of the matrix, Rows = books, Columns = users, Now KNN learns : similarity between BOOKS
    # index is book title then this is item-item based collaborative filtering
    book_pivot = train_ratings.pivot_table(
        index="title",
        columns="user_id",
        values="ratings"
    ).fillna(0)
    
    # print(book_pivot.shape)
    # book_pivot.head(3)

    return book_pivot, train_ratings, test_ratings

### Function to Train KNN with Parameters

In [17]:
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

def train_knn(book_pivot, metric, n_neighbors):
    knn_model = NearestNeighbors(
        metric=metric,
        algorithm='brute',
        n_neighbors=n_neighbors # Model setting (max neighbors model is allowed to search while traning)
    )

    book_sparse = csr_matrix(book_pivot)
    # print("book_sparse.shape : ", book_sparse.shape)
    # print("book_sparse.nnz : ", book_sparse.nnz) # So out of 636,636 possible cells, only : 11,819 ratings actually exist

    knn_model.fit(book_sparse)
    print("Model trained succussfully")
    return knn_model, n_neighbors

### Recommendation function

In [18]:
def recommend_books_knn(seed_book:str, n_suggestions: int=5):
        
    # Skip unknown books
    if seed_book not in book_pivot.index:
        return []

    book_idx = book_pivot.index.get_loc(seed_book) # At what row number is this book located?

    # knn_model can give prediction less thatn it trainded numober of neighbors
    if n_neighbors_trained < n_suggestions:
        n_suggestions = n_neighbors_trained-1
        print("Warning : n_suggestions > trained neighbors, set to max allowed:", n_suggestions)
    
    distance, indices = knn_model.kneighbors(
        # get reshaped row vector of a particular book by id that converted into numpy array from pandas series
        # since a book data in a row form not in column, So each book must be a row >> (1, -1)
        book_pivot.iloc[book_idx, : ].values.reshape(1, -1), 

        # keep less than n_neighbors in training time. How many recommendations we want to return in prediction time
        # Since closest book is the book itself add one more book k+1
        # skip the book itself later
        n_neighbors=n_suggestions+1 
    )
        
    # rec_books = []
    # for i in range(1, len(indices.flatten())):
    #     rec_books.append(book_pivot.index[indices.flatten()[i]])

    rec_books = [book_pivot.index[i] for i in indices.flatten()[1:]]
    # print(rec_books)
    return rec_books

#### Smaple test

In [22]:
# This knn_model work as follows
# if some one open a book A knn_model look for books that in same cluster with min distance. The cluster is created using ratings of each user.
# If users read/rated book A, Book B and also Book C, then another user open/click on Book A, then we can say/ suggest some 
# other books that in same cluster with Book A with min distance to Book A, cluster creation done by actualy according to user interactions 
# here only ratings in content based system we use genre, title, category, ... but in collaborative approach we use user interaction to cluster 
# simillar vibe books

seed_book = "A Bend in the Road"
n_neighbors_trained = 5

book_pivot, train_ratings, test_ratings = build_dataset(
    min_user_ratings=200, 
    min_book_ratings=50
)

knn_model, n_neighbors = train_knn(
    book_pivot= book_pivot,
    metric ="cosine", 
    n_neighbors= 5
)
book_pivot
# automaticaly rec function will fetch need other varibles no need to pass as parameters such as,  book_pivot, n_neighbors_trained
recs = recommend_books_knn(
    seed_book=seed_book, 
    n_suggestions=5, 
)

recs

Model trained succussfully


['Nights in Rodanthe',
 'By the Light of the Moon',
 'Angels',
 'A Walk to Remember',
 'The Last Time They Met : A Novel']

## **Model Evaluation**
- When we recommend books to users, how often are they actually books the user likes?
#### **evaluation must be user-based :**
- We simulate real world:

1. Hide some ratings from users
2. Recommend books to that user
3. Check if recommended books were actually liked

- This is called Train/Test split for recommender systems.


  
#### **Precision@K Intuition**
- **how accurate the recommendations are**

- Imagine we recommend K = 5 books to a user.
- Recommended → [A, B, C, D, E]
- Actual liked books → [A, C, F]
- Relevant recommendations = A, C = 2
- Precision@5 = relevant / recommended
 = 2 / 5 = 0.40
-  “40% of recommendations were good”



#### **Recall@K Intuition**

- **how much of the user’s liked books we found**

- User liked 3 books → [A, C, F]
- We found only [A, C]

- Recall@5 = found relevant / total relevant
= 2 / 3 = 0.67

-  “We found 67% of books user likes”

In [28]:
# k = how many books we recommend
# rate_threshold = rating considered “liked”

def precision_recall_model(test_ratings, n_suggestions=5, rate_threshold=8):

    # We will compute precision/recall for each user, then average them
    precisions, recalls = [], []

    # Evaluate recommendations user by user final get avg
    # So if test_ratings contains 100 users, you get 100 recommend_books function calls for that one trained knn_model
    # Multiply that by 108 models > thousands of printed lists if you had multiple users in each dataset
    for user in test_ratings["user_id"].unique():

        # This dataframe contains : books this user rated in the TEST set
        user_test = test_ratings[test_ratings["user_id"] == user]
        # books the user actually liked
        liked_books = user_test[user_test["ratings"] >= rate_threshold]["title"].tolist()

        # Skip users who didn’t like anything
        if len(liked_books) == 0:
            continue
        
        # pick one book the user liked (seed book)
        seed_book = liked_books[0]

        # Generate recommendations
        recs = recommend_books_knn(
            seed_book=seed_book, 
            n_suggestions=n_suggestions, 
        )

        # Skip if no recs
        if len(recs) == 0:
            continue

        # Find relevant recommendations
        # We compute intersection : recommended books ∩ liked books
        # recs        = [A, B, C, D, E]
        # liked_books = [B, D, F]
        # intersection = [B, D]
        # relevant = 2
        relevant = len(set(recs) & set(liked_books))

        # Compute Precision@K & Recall@K
        # Store user results, We repeat for all users
        precisions.append(relevant / n_suggestions) # Out of recommended books, how many were good?
        recalls.append(relevant / len(liked_books)) # “Out of all books the user likes, how many did we find?”

    # Return overall performance, We average across users → final score
    return np.mean(precisions), np.mean(recalls)

### Grid Search

In [ ]:
import itertools

# We import a tool that helps us try all combinations of parameters automatically.
# for u, b, m, n in itertools.product(user_params, book_params, metrics, neighbors): >>> Generate ALL combinations
# It’s like nested loops:
# for u in user_params:
#  for b in book_params:
#    for m in metrics:
#      for n in neighbors:

n_neighbors_trained = 5

user_params = [50, 100, 200]
book_params = [50, 100, 200]
metrics = ['cosine', 'euclidean', 'manhattan'] # Distance formulas
neighbors =[5, 10, 20, 30] # How many nearest neighbors 


"""results looked like this:
[
 [50, 50, 'cosine', 10, 0.31, 0.22],
 [50,100, 'euclidean', 5, 0.28, 0.20],
 ...
]
"""
results = [] 
count = 0

for u, b, m, n in itertools.product(user_params, book_params, metrics, neighbors): # Generate ALL combinations
    # print(f"Testing : users>{u}, books>{b}, metric={m}, neighbors={n}")
    
    count+=1
    
    book_pivot, train_ratings, test_ratings = build_dataset(u, b)

    if book_pivot.shape[0] < 10:
        continue
        
    print(f"Processing ({count}): users>{u}, books>{b}, metric={m}, neighbors={n}")

    knn_model, n_neighbors = train_knn(
        book_pivot, 
        m, 
        n
    )
    
    precision, recall = precision_recall_model(
        test_ratings=test_ratings,
        n_suggestions=5,
        rate_threshold=8,
    )

    results.append([u, b, m, n, precision, recall])

### **Show Best Config**

In [19]:
results_df = pd.DataFrame(results, columns=[
    "min_user_ratings",
    "min_book_ratings",
    "metric",
    "neighbors",
    "precision",
    "recall"
])

# Sorts the table by precision column
# ascending=False → highest precision first (best models on top)
# Show top 10 models, show the 10 best parameter combinations
results_df.sort_values("precision", ascending=False).head(10)

NameError: name 'results' is not defined

## **Best model**

In [29]:
best_min_user_ratings = 200
best_min_book_ratings = 100
best_metric = "cosine"
best_n_neighbors = 30

seed_book = "A Bend in the Road"
n_neighbors_trained = 5

book_pivot, train_ratings, test_ratings = build_dataset(
    min_user_ratings=best_min_user_ratings, 
    min_book_ratings=best_min_book_ratings
)

knn_model, n_neighbors = train_knn(
    book_pivot= book_pivot,
    metric = best_metric, 
    n_neighbors= best_n_neighbors
)

recs = recommend_books_knn(
    seed_book=seed_book, 
    n_suggestions=5, 
)

precision, recall = precision_recall_model(
    test_ratings, 
    n_suggestions=5, 
    rate_threshold=8, 
)

print(recs)
print(f"\nPrecission : {precision}, Recall : {recall}")

Model trained succussfully
['Circle of Friends', 'Cradle and All', 'Good in Bed', 'House of Sand and Fog', 'A Map of the World']

Precission : 0.016956521739130436, Recall : 0.029222080136402387


## **Save KNN model as pkls**

In [30]:
import joblib
import os

# Create dir to save pkls
directory = "knn_pkls"
if not os.path.exists(directory):
    os.mkdir(directory)

# Save KNN model into knn_pkls folder
joblib.dump(knn_model, f"{directory}/knn_model.pkl")
train_ratings.to_csv(f"{directory}/train_ratings_knn.csv", index=False)

joblib.dump(book_pivot,  f"{directory}/book_pivot_knn.joblib")

['knn_pkls/book_pivot_knn.joblib']

## **Popularity-Based Recommender Model**

In [55]:
# train_ratings → a dataframe containing user ratings
# min_ratings=50 → default threshold.
#  book_pivot, train_ratings_10_50, test_ratings_10_50

# avg_rating => ∑ri / N = sum of all ratings of a book / number of ratings ( 0 also get into sum & count)
# score => avg_rating × N = sum of all rarings of a particular book


def recommend_books_popular(n_suggestions: int =5):

    global popularity
    
    # create a new table that summarizes ratings for each book
    popularity = train_ratings.groupby("title") # We haven’t calculated anything yet, We only separated rows into buckets
    print("1 : ", type(popularity))
    
    # "rating" = This is the column name in your DataFrame, "count" & "mean" = These are aggregation functions that Pandas already knows
    # “For each group, calculate these statistics from the rating column" > count -How many ratings exist, mean - Average rating
    # "count" & "mean" are built-in Pandas aggregation functions used inside .agg()
    popularity = popularity.agg({
        "ratings" : ["count", "mean"]
    })
    print("2 : ", type(popularity))

    # After aggregation, pandas creates multi-level column rename them to simpler names 
    popularity.columns = ["rating_count", "avg_rating"]

    # Create ranking score => average_rating × number_of_ratings
    popularity["score"] = (
        popularity["avg_rating"] * popularity["rating_count"]
    )

    popularity = popularity.sort_values("score", ascending=False)
    popularity = popularity.reset_index() 
    
    return popularity['title'].head(n_suggestions).tolist()

## Evaluate Popularity Model

In [56]:
def evaluate_popularity(n_suggestions =5, rate_threshold=8):
    
    precisions, recalls = [], []

    popular_books = recommend_books_popular(n_suggestions=5)
    
    for user in test_ratings_10_50["user_id"].unique():
        
        user_test = test_ratings_10_50[test_ratings_10_50["user_id"] == user] # get records for a particular user
        liked_books = user_test[user_test["ratings"] >= rate_threshold]["title"].tolist() # list of book titles which rated more than rate_threshold value

        if len(liked_books) == 0:
            continue

        relevant = len(set(popular_books) & set(liked_books)) # get intersectoin for find correct predictions

        # We need corrects we only got relevant no of books
        precision = relevant / n_suggestions

        recall = relevant / len(liked_books) # User liked len of liked_books no of books, but model were able to find only relevant no of books only

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

In [57]:
# Popularity model does NOT need heavy user filtering

_, train_ratings_10_50, test_ratings_10_50 = build_dataset(min_user_ratings=10, min_book_ratings=50)

popular_books = recommend_books_popular(n_suggestions=5)

popular_books

1 :  <class 'pandas.core.groupby.generic.DataFrameGroupBy'>
2 :  <class 'pandas.core.frame.DataFrame'>


['The Lovely Bones: A Novel',
 'Harry Potter and the Chamber of Secrets (Book 2)',
 'The Da Vinci Code',
 'To Kill a Mockingbird',
 'Harry Potter and the Prisoner of Azkaban (Book 3)']

In [58]:
pop_precision, pop_recall = evaluate_popularity(n_suggestions=5)

print("Popularity Precision@5:", pop_precision)
print("Popularity Recall@5:", pop_recall)

1 :  <class 'pandas.core.groupby.generic.DataFrameGroupBy'>
2 :  <class 'pandas.core.frame.DataFrame'>
Popularity Precision@5: 0.009955423476968797
Popularity Recall@5: 0.024574787060146732


## **Precission => KNN model beats the baseline**
- Knn recommender is better than popularity base reccommendation, i.e) Instead of recommending popular books for each user we can recommned books using a knn model

In [59]:
improvement = (precision - pop_precision) / pop_precision
print(f"Improvement Of Precission : {improvement*100:.2f} %")

Improvement Of Precission : 70.32 %


## **Save popularty df as csv**

In [61]:
popularity.head()

,title,rating_count,avg_rating,score
0,The Lovely Bones: A Novel,225,3.355556,755.0
1,Harry Potter and the Chamber of Secrets (Book 2),141,3.978723,561.0
2,The Da Vinci Code,182,2.901099,528.0
3,To Kill a Mockingbird,138,3.644928,503.0
4,Harry Potter and the Prisoner of Azkaban (Book 3),109,4.394495,479.0


In [60]:
import os
import pandas as pd

# Create dir to save csv
directory = "popularity_csv"

if not os.path.exists(directory):
    os.mkdir(directory)

popularity.to_csv(f"{directory}/popularity_df.csv", index=False)